# 🫁 LUNA16 Exploratory Data Analysis

This notebook walks through the LUNA16 dataset to understand:
1. CT volume properties (size, spacing, HU distributions)
2. Nodule annotation statistics (sizes, locations)
3. Preprocessing effects
4. Data imbalance characteristics

**Run order:** Preprocess the data first (`ct_preprocessing.py`), then execute all cells.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import SimpleITK as sitk
from tqdm import tqdm

# Project imports
from preprocessing.ct_preprocessing import load_ct_volume, apply_hu_windowing

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
print('✓ Imports successful')

## 1. Configuration

In [ ]:
# ── Paths — modify to your local setup ──────────────────────────────────────
LUNA16_RAW_DIR      = Path('/data/LUNA16')
LUNA16_PREP_DIR     = Path('/data/LUNA16/preprocessed')
ANNOTATIONS_CSV     = LUNA16_RAW_DIR / 'annotations.csv'

# Load annotations
annotations = pd.read_csv(ANNOTATIONS_CSV)
print(f'Total annotated nodules: {len(annotations)}')
print(f'Unique CT scans:         {annotations["seriesuid"].nunique()}')
annotations.head()

## 2. Nodule Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Diameter histogram
axes[0].hist(annotations['diameter_mm'], bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(annotations['diameter_mm'].median(), color='red', linestyle='--', label=f'Median: {annotations["diameter_mm"].median():.1f} mm')
axes[0].set_xlabel('Nodule Diameter (mm)')
axes[0].set_ylabel('Count')
axes[0].set_title('Nodule Diameter Distribution')
axes[0].legend()

# Size categories
size_bins = [3, 6, 10, 20, 50]
size_labels = ['3–6mm\n(small)', '6–10mm\n(medium)', '10–20mm\n(large)', '>20mm\n(very large)']
size_counts = pd.cut(annotations['diameter_mm'], bins=size_bins, labels=size_labels).value_counts().sort_index()
axes[1].bar(range(len(size_counts)), size_counts.values, color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
axes[1].set_xticks(range(len(size_counts)))
axes[1].set_xticklabels(size_counts.index, fontsize=9)
axes[1].set_ylabel('Count')
axes[1].set_title('Nodule Size Categories')

# Nodules per scan
nodules_per_scan = annotations.groupby('seriesuid').size()
axes[2].hist(nodules_per_scan, bins=range(1, nodules_per_scan.max()+2), 
             color='mediumpurple', edgecolor='white', linewidth=0.5, align='left')
axes[2].set_xlabel('Nodules per CT Scan')
axes[2].set_ylabel('Number of Scans')
axes[2].set_title('Nodules per Scan Distribution')

plt.suptitle('LUNA16 Nodule Annotation Statistics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('luna16_nodule_stats.png', bbox_inches='tight')
plt.show()

print(f'Diameter stats:\n{annotations["diameter_mm"].describe()}')

## 3. CT Volume Properties

In [ ]:
# Sample a few scans to analyse volume properties
n_sample = 20
mhd_files = list(LUNA16_RAW_DIR.rglob('*.mhd'))[:n_sample]

volume_stats = []
for mhd_path in tqdm(mhd_files, desc='Scanning volumes'):
    try:
        image = sitk.ReadImage(str(mhd_path))
        size = image.GetSize()       # (W, H, D) in sitk convention
        spacing = image.GetSpacing() # (x, y, z) in mm
        volume_stats.append({
            'seriesuid': mhd_path.stem,
            'width': size[0], 'height': size[1], 'depth': size[2],
            'spacing_x': spacing[0], 'spacing_y': spacing[1], 'spacing_z': spacing[2],
        })
    except Exception as e:
        print(f'Error loading {mhd_path.name}: {e}')

stats_df = pd.DataFrame(volume_stats)
print('\nVolume shape statistics:')
print(stats_df[['width', 'height', 'depth']].describe().round(1))
print('\nVoxel spacing statistics (mm):')
print(stats_df[['spacing_x', 'spacing_y', 'spacing_z']].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# In-plane spacing distribution
axes[0].scatter(stats_df['spacing_x'], stats_df['depth'], alpha=0.6, s=40, color='steelblue')
axes[0].set_xlabel('In-plane Spacing (mm)')
axes[0].set_ylabel('Depth (number of slices)')
axes[0].set_title('In-plane Spacing vs. Slice Count')
axes[0].axvline(1.0, color='red', linestyle='--', alpha=0.7, label='1mm target')
axes[0].legend()

# Slice thickness distribution
axes[1].hist(stats_df['spacing_z'], bins=20, color='salmon', edgecolor='white')
axes[1].set_xlabel('Slice Thickness (mm)')
axes[1].set_ylabel('Count')
axes[1].set_title('Slice Thickness Distribution')
axes[1].axvline(1.0, color='darkred', linestyle='--', label='1mm target')
axes[1].legend()

# Volume dimensions
axes[2].scatter(stats_df['width'], stats_df['depth'], alpha=0.6, s=40, color='mediumseagreen')
axes[2].set_xlabel('Width (pixels)')
axes[2].set_ylabel('Depth (slices)')
axes[2].set_title('Volume Dimensions')

plt.suptitle('CT Volume Properties (sample of 20 scans)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. HU Distribution Analysis

In [ ]:
# Load a single scan to analyse HU values
sample_mhd = mhd_files[0]
volume, spacing, origin = load_ct_volume(str(sample_mhd))

print(f'Volume shape: {volume.shape}')
print(f'HU range: [{volume.min():.0f}, {volume.max():.0f}]')
print(f'Voxel spacing: {spacing} mm')

# HU value distribution (subsample for speed)
sample_voxels = volume.ravel()[::100]   # Every 100th voxel

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full HU histogram
axes[0].hist(sample_voxels, bins=200, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('Hounsfield Units (HU)')
axes[0].set_ylabel('Voxel Count (sampled)')
axes[0].set_title('Full HU Distribution')

# Annotate tissue types
tissue_labels = [
    (-1000, 'Air\n(-1000 HU)', 'lightyellow'),
    (-700, 'Lung\n(-700 HU)', 'lightblue'),
    (0, 'Water\n(0 HU)', 'lightgreen'),
    (100, 'Soft tissue\n(+50 HU)', 'lightsalmon'),
    (700, 'Bone\n(+700 HU)', 'lightyellow'),
]
for hu, label, color in tissue_labels:
    axes[0].axvline(hu, color='gray', linestyle=':', alpha=0.5)

# Lung window region (what the model sees)
axes[0].axvspan(-1000, 400, alpha=0.1, color='orange', label='Lung window [-1000, +400]')
axes[0].legend(loc='upper right')

# Windowed & normalised
volume_norm = apply_hu_windowing(volume, hu_min=-1000, hu_max=400)
sample_norm = volume_norm.ravel()[::100]
axes[1].hist(sample_norm, bins=100, color='mediumseagreen', alpha=0.8, edgecolor='none')
axes[1].set_xlabel('Normalised Value [0, 1]')
axes[1].set_ylabel('Voxel Count (sampled)')
axes[1].set_title('After HU Windowing + Normalisation')

plt.suptitle(f'HU Analysis: {sample_mhd.stem[:30]}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Visualise a Sample CT Scan with Nodule Mask

In [ ]:
# Find a scan that has annotated nodules
scans_with_nodules = annotations['seriesuid'].unique()

# Look for matching preprocessed file
sample_uid = None
sample_img = sample_mask = None

for subset_dir in sorted(LUNA16_PREP_DIR.glob('subset*')):
    for img_path in subset_dir.glob('*_image.npy'):
        uid = img_path.stem.replace('_image', '')
        if uid in scans_with_nodules:
            mask_path = subset_dir / f'{uid}_mask.npy'
            if mask_path.exists():
                sample_uid = uid
                sample_img = np.load(str(img_path))
                sample_mask = np.load(str(mask_path))
                break
    if sample_uid:
        break

if sample_uid is None:
    print('No preprocessed scans found. Run ct_preprocessing.py first.')
else:
    print(f'Series UID: {sample_uid}')
    print(f'Image shape: {sample_img.shape}')
    print(f'Mask shape: {sample_mask.shape}')
    print(f'Nodule voxels: {sample_mask.sum()} ({100*sample_mask.mean():.4f}%)')
    
    # Find slices with nodule tissue
    nodule_slices = np.where(sample_mask.sum(axis=(1,2)) > 0)[0]
    print(f'Nodule present in slices: {nodule_slices.tolist()}')

In [ ]:
if sample_uid is not None and len(nodule_slices) > 0:
    # Show 5 slices around the nodule
    centre_slice = nodule_slices[len(nodule_slices)//2]
    show_slices = np.linspace(
        max(0, centre_slice - 8),
        min(sample_img.shape[0]-1, centre_slice + 8),
        5, dtype=int
    )
    
    fig, axes = plt.subplots(2, 5, figsize=(18, 7))
    
    for col, sl in enumerate(show_slices):
        ct_sl = sample_img[sl]
        mask_sl = sample_mask[sl]
        
        # Top row: raw CT
        axes[0, col].imshow(ct_sl, cmap='gray', vmin=0, vmax=1)
        axes[0, col].set_title(f'CT Slice {sl}', fontsize=9)
        axes[0, col].axis('off')
        
        # Bottom row: CT + mask overlay
        axes[1, col].imshow(ct_sl, cmap='gray', vmin=0, vmax=1)
        if mask_sl.sum() > 0:
            # Create RGBA overlay
            overlay = np.zeros((*mask_sl.shape, 4))
            overlay[mask_sl > 0] = [0.0, 1.0, 0.0, 0.5]  # Green
            axes[1, col].imshow(overlay)
        has_nodule = '⬤' if mask_sl.sum() > 0 else '○'
        axes[1, col].set_title(f'{has_nodule} +Mask [{sl}]', fontsize=9)
        axes[1, col].axis('off')
    
    plt.suptitle(f'CT Scan with Nodule Mask Overlay\n{sample_uid[:40]}',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig('sample_ct_with_mask.png', bbox_inches='tight')
    plt.show()

## 6. Class Imbalance Analysis

In [ ]:
if sample_uid is not None:
    total_voxels = sample_mask.size
    positive_voxels = sample_mask.sum()
    negative_voxels = total_voxels - positive_voxels
    imbalance_ratio = negative_voxels / positive_voxels
    
    print('Class Imbalance Analysis')
    print('=' * 40)
    print(f'Total voxels:     {total_voxels:>12,}')
    print(f'Nodule voxels:    {positive_voxels:>12,}  ({100*positive_voxels/total_voxels:.4f}%)')
    print(f'Background:       {negative_voxels:>12,}  ({100*negative_voxels/total_voxels:.4f}%)')
    print(f'Imbalance ratio:  {imbalance_ratio:>12,.0f}:1 (background:nodule)')
    print()
    print('→ This extreme imbalance is why Dice loss + positive weighting')
    print('  is critical. BCE alone would converge to all-background prediction.')

    # Visualise as pie chart
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        [negative_voxels, positive_voxels],
        labels=[f'Background\n({100*negative_voxels/total_voxels:.3f}%)',
                f'Nodule\n({100*positive_voxels/total_voxels:.3f}%)'],
        colors=['#5B9BD5', '#ED7D31'],
        startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    ax.set_title('Voxel Class Distribution\n(Extreme Imbalance)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('class_imbalance.png', bbox_inches='tight')
    plt.show()

## 7. Data Augmentation Preview

In [ ]:
from augmentation.medical_augmentations import (
    RandomRotation3D, RandomFlip3D, GaussianNoise, IntensityScale, ElasticDeformation3D
)

if sample_uid is not None and len(nodule_slices) > 0:
    # Extract a 64³ patch centred on the nodule
    cz, cy, cx = (
        nodule_slices[len(nodule_slices)//2],
        sample_img.shape[1]//2,
        sample_img.shape[2]//2
    )
    
    from datasets.luna16_dataset import extract_patch
    patch_img, _ = extract_patch(sample_img, (cz, cy, cx), (64, 64, 64))
    patch_mask, _ = extract_patch(sample_mask.astype(np.float32), (cz, cy, cx), (64, 64, 64))
    
    # Add channel dim: [1, D, H, W]
    img_ch = patch_img[np.newaxis]
    mask_ch = patch_mask[np.newaxis]
    
    # Define transforms to preview
    transforms = [
        ('Original', None),
        ('Rotation', RandomRotation3D((-20, 20), prob=1.0)),
        ('Flip', RandomFlip3D(prob=1.0)),
        ('Noise', GaussianNoise(std_range=(0.03, 0.05), prob=1.0)),
        ('Intensity Scale', IntensityScale(scale_range=(0.8, 1.2), prob=1.0)),
        ('Elastic', ElasticDeformation3D(num_control_points=5, max_displacement=10, prob=1.0)),
    ]
    
    centre_slice_local = 32  # Centre of 64³ patch
    fig, axes = plt.subplots(2, len(transforms), figsize=(3.2*len(transforms), 7))
    
    for col, (name, transform) in enumerate(transforms):
        if transform is None:
            aug_img, aug_mask = img_ch, mask_ch
        else:
            aug_img, aug_mask = transform(img_ch.copy(), mask_ch.copy())
        
        ct_sl = aug_img[0, centre_slice_local]
        mask_sl = aug_mask[0, centre_slice_local]
        
        axes[0, col].imshow(ct_sl, cmap='gray', vmin=0, vmax=1)
        axes[0, col].set_title(name, fontsize=9, fontweight='bold')
        axes[0, col].axis('off')
        
        axes[1, col].imshow(ct_sl, cmap='gray', vmin=0, vmax=1)
        if mask_sl.sum() > 0:
            overlay = np.zeros((*mask_sl.shape, 4))
            overlay[mask_sl > 0.5] = [0.0, 1.0, 0.0, 0.5]
            axes[1, col].imshow(overlay)
        axes[1, col].axis('off')
    
    axes[0, 0].set_ylabel('CT Slice', fontsize=10)
    axes[1, 0].set_ylabel('+Mask', fontsize=10)
    
    plt.suptitle('Data Augmentation Preview (Centre Axial Slice)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('augmentation_preview.png', bbox_inches='tight')
    plt.show()

## Summary

Key findings from the EDA:
1. **Nodule size range**: 3–30mm, median ~6mm — most nodules are small
2. **Scanner variability**: spacing varies from 0.5–2.5mm; resampling to 1mm³ is essential
3. **Class imbalance**: nodule voxels ≈ 0.08% of total — Dice loss is critical
4. **HU windowing**: lung window [-1000, +400] removes irrelevant high-density structures
5. **Augmentation**: rotation and elastic deformation effectively diversify the training set